In [1]:
from transformers import GPT2LMHeadModel, GPT2TokenizerFast, GPT2Config
from transformers import get_linear_schedule_with_warmup

import torch
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import random_split, RandomSampler, SequentialSampler

import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
# model_name: ['gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl']
model_name = "gpt2-large" 
model_save_path = './model'

/u1/kfountou/.conda/envs/the_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
configuration = GPT2Config.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name, config=configuration)

tokenizer = GPT2TokenizerFast.from_pretrained(model_name)

model = model.to(device)

In [3]:
from nlp_dataset import generate_sample

def form_string(sample: tuple[list, float], isTrain: bool) ->tuple[str, float]:
    sample_text = sample[0]
    sample_ans = sample[1]
    input_list = [x if type(x) == str else format(x, '05.2f') for x in sample_text[:-1]]
    prompt = "<|startoftext|>" + ", ".join(input_list) + ". " + sample_text[-1] + "."
    if isTrain:
        prompt += " Answer: " + format(sample_ans, '05.2f')
        prompt += "<|endoftext|>"
    # else:
    #     prompt += " Answer: "
    return prompt, sample_ans

In [ ]:
num_cats = 8
query_type = "min"
num_query_cats = 4
train_low = 0
train_high = 5
test_low = 0
test_high = 20
num_train_samples = 1000000
num_test_samples = 100
num_val_samples = 100

In [5]:
train_data_comb = [form_string(generate_sample(num_cats, query_type, train_low, train_high, num_query_cats), True) for _ in range(num_train_samples)]
train_data = [x[0] for x in train_data_comb]
train_data_ans = [x[1] for x in train_data_comb]
test_data_comb = [form_string(generate_sample(num_cats, query_type, test_low, test_high, num_query_cats, train=False), False) for _ in range(num_test_samples)]
test_data = [x[0] for x in test_data_comb]
test_data_ans = [x[1] for x in test_data_comb]
val_data_comb = [form_string(generate_sample(num_cats, query_type, train_low, train_high, num_query_cats, train=True), False) for _ in range(num_val_samples)]
val_data = [x[0] for x in val_data_comb]
val_data_ans = [x[1] for x in val_data_comb]

In [6]:
print(val_data[0])
print(val_data_ans[0])

<|startoftext|>Cat-ȇ, Cat-ȃ, Catǹ, 00.65, CatǼ, 01.81, Catǽ, 02.14, CatǾ, 01.42, Catȃ, 01.19, CatȄ, 02.33, Catȅ, 01.20, Catȇ, 00.44. Find min of categories Catȃ, Catȅ, CatǾ and CatǼ.
1.19


In [7]:
tokenizer = GPT2TokenizerFast.from_pretrained(model_name,
                                              bos_token='<|startoftext|>',
                                              eos_token='<|endoftext|>',
                                              unk_token='<|unknown|>',
                                              pad_token='<|pad|>'
                                             )

In [8]:
batch_size = 16
max_length = 101

# standard PyTorch approach of loading data in using a Dataset class.
class NAR_Dataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.input_ids = []
        self.attn_masks = []

        for data_point in data:
            encodings = tokenizer.encode_plus(data_point,
                                              truncation=True,
                                              padding='max_length',
                                              max_length=max_length,
                                              # return a PyTorch tensor
                                              return_tensors='pt'       
                                             )
            self.input_ids.append(torch.squeeze(encodings['input_ids'],0))
            self.attn_masks.append(torch.squeeze(encodings['attention_mask'],0))


    def __len__(self):
        return len(self.data)

    def __getitem__(self,idx):
        return self.input_ids[idx], self.attn_masks[idx]

dataset_indist_train = NAR_Dataset(train_data, tokenizer)
dataset_indist_val = NAR_Dataset(val_data, tokenizer)
dataset_ood = NAR_Dataset(test_data, tokenizer)
print(f"input_ids: {dataset_ood[0][0]} attn_masks: {dataset_ood[0][1]}")

input_ids: tensor([50257, 21979,    62,   132,   237,    11,  5181,   132,   232,    11,
         1367,    13,  3865,    11,  5181,   132,   233,    11,  1467,    13,
         2154,    11,  5181,   132,   237,    11,  1367,    13,  2425,    11,
         5181,   132,   240,    11,   838,    13,  3865,    11,  5181,   132,
          242,    11,  1105,    13,  3270,    11,  5181,   132,   244,    11,
         5181,    62,   132,   244,    11,  8702,    13,  3270,    11,  5181,
          132,   245,    11,  1367,    13,  6469,    11,  5181,   132,   246,
           11,  8487,    13,  3365,    13,  9938,   949,   286,  9376,  5181,
          132,   232,    11,  5181,   132,   233,    11,  5181,   132,   245,
          290,  5181,   132,   237,    13, 50259, 50259, 50259, 50259, 50259,
        50259]) attn_masks: tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1

In [9]:
print(tokenizer.decode(dataset_indist_train[0][0]))

<|startoftext|>CatǶ, 03.07, Catǹ, 03.16, Cat-Ȃ, Cat-Ƕ, CatǼ, 03.20, Catǽ, 03.12, Catǿ, 03.25, Catȁ, 03.12, CatȂ, 03.28, CatȆ, 03.14. Find min of categories CatǶ, CatȆ, Catǹ and CatǼ. Answer: 03.07<|endoftext|>


In [10]:
print(tokenizer.decode(dataset_indist_train[10][0]))

<|startoftext|>CatǸ, 02.69, Catǿ, 02.75, CatȀ, Cat+ȁ, 02.75, Cat+Ȉ, Catȁ, 02.64, CatȄ, 02.66, Catȅ, 02.61, CatȆ, 02.65, CatȈ, 02.58. Find min of categories Catȅ, Catȁ, CatȆ and Catǿ. Answer: 02.61<|endoftext|>


In [11]:

train_dataloader = DataLoader(
            dataset_indist_train, 
            sampler = RandomSampler(dataset_indist_train),
            batch_size = batch_size # Trains with this batch size.
        )

# Get valiation samples sequentially.
validation_dataloader = DataLoader(
            dataset_indist_val, 
            sampler = SequentialSampler(dataset_indist_val),
            batch_size = batch_size # Evaluate with this batch size.
        )

test_dataloader = DataLoader(
            dataset_ood, 
            sampler = SequentialSampler(dataset_ood),
            batch_size = batch_size # Evaluate with this batch size.
        )
            


In [12]:
configuration = GPT2Config.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name, config=configuration)
model = model.to(device)
model.resize_token_embeddings(len(tokenizer))

epochs = 3
learning_rate = 2e-5
warmup_steps = 1e2
# to prevent any division by zero in the implementation
epsilon = 1e-8
optim = AdamW(model.parameters(), lr = learning_rate, eps = epsilon)

total_steps = len(train_dataloader) * epochs  # [no batches] x [no epochs]

# Create the learning rate scheduler.
scheduler = get_linear_schedule_with_warmup(optim,
                                            num_warmup_steps=warmup_steps,
                                            num_training_steps=total_steps)

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [13]:
def infer(prompt):
    input = prompt
    input = tokenizer(input, return_tensors="pt")
    input_ids      = input["input_ids"]
    attention_mask = input["attention_mask"]

    output = model.generate(input_ids.to(device),
                            attention_mask=attention_mask.to(device),
                            max_new_tokens=5,
                            do_sample = True, top_k = 1, top_p = 0.85, pad_token_id=tokenizer.eos_token_id)
    output = tokenizer.decode(output[0], skip_special_tokens=True)
    start_index = output.find("Answer: ") + len("Answer: ")
    # Extract the first 5 characters from that point
    result = output[start_index:start_index + 5]

    return result

In [14]:
def infer2(prompt):
    input = prompt
    input = tokenizer(input, return_tensors="pt")
    input_ids      = input["input_ids"]
    attention_mask = input["attention_mask"]

    output = model.generate(input_ids.to(device),
                            attention_mask=attention_mask.to(device),
                            max_new_tokens=5,
                            do_sample = True, top_k = 1, top_p = 0.85, pad_token_id=tokenizer.eos_token_id)
    output = tokenizer.decode(output[0], skip_special_tokens=True)

    return output

In [15]:
import numpy as np

def get_metrics(predictions, target):
    diff, off = [], 0
    for (i, pred) in enumerate(predictions):
        try:
            diff.append(abs(float(pred) - target[i])**2)
        except:
            off += 1
    
    if len(diff) == 0:
        return np.inf, off / len(predictions) * 100
    
    return sum(diff)/len(diff), off / len(predictions) * 100
    

In [ ]:
for epoch_i in range(0, epochs):
    total_train_loss = 0
    model.train() 

    for step, batch in enumerate(train_dataloader): 
        b_input_ids = batch[0].to(device) 
        b_labels    = batch[0].to(device)
        b_masks     = batch[1].to(device) 

        model.zero_grad()
        outputs = model( input_ids = b_input_ids, labels = b_labels,
                         attention_mask = b_masks, token_type_ids = None )

        loss = outputs[0]

        # Get sample every x batches.
        if step % 100 == 0 and not step == 0:
            model.eval()
            test_preds = [infer(test_data[i]) for i in range(len(test_data))]
            test_metrics = get_metrics(test_preds, test_data_ans)
            print(f"Test Loss: {test_metrics[0]} Test Off: {test_metrics[1]}")
            val_preds = [infer(val_data[i]) for i in range(len(val_data))]
            val_metrics = get_metrics(val_preds, val_data_ans)
            print(f"Val Loss: {val_metrics[0]} Val Off: {val_metrics[1]}")
            print(infer2(val_data[55]))
            if val_metrics[0] < 0.01:
                break
            model.train()

        loss.backward()
        optim.step()
        scheduler.step()

Test Loss: 3.2729120000000016 Test Off: 0.0
Val Loss: 0.11449200000000001 Val Off: 0.0
CatǶ, 01.02, CatǸ, 01.79, CatǺ, 01.29, Catǿ, Cat+Ȅ, Cat+Ǹ, 01.18, CatȂ, 01.73, CatȄ, 00.92, Catȅ, 01.56, Catȇ, 01.65. Find min of categories Catȅ, CatǸ, Catǿ and CatǶ. Answer: 00.92
Test Loss: 3.4267957894736853 Test Off: 5.0
Val Loss: 0.11400699999999998 Val Off: 0.0
CatǶ, 01.02, CatǸ, 01.79, CatǺ, 01.29, Catǿ, Cat+Ȅ, Cat+Ǹ, 01.18, CatȂ, 01.73, CatȄ, 00.92, Catȅ, 01.56, Catȇ, 01.65. Find min of categories Catȅ, CatǸ, Catǿ and CatǶ. Answer: 00.92
Test Loss: 6.530821000000002 Test Off: 0.0
Val Loss: 0.12038300000000002 Val Off: 0.0
CatǶ, 01.02, CatǸ, 01.79, CatǺ, 01.29, Catǿ, Cat+Ȅ, Cat+Ǹ, 01.18, CatȂ, 01.73, CatȄ, 00.92, Catȅ, 01.56, Catȇ, 01.65. Find min of categories Catȅ, CatǸ, Catǿ and CatǶ. Answer: 00.92
Test Loss: 5.849096907216496 Test Off: 3.0
Val Loss: 0.09604599999999999 Val Off: 0.0
CatǶ, 01.02, CatǸ, 01.79, CatǺ, 01.29, Catǿ, Cat+Ȅ, Cat+Ǹ, 01.18, CatȂ, 01.73, CatȄ, 00.92, Catȅ, 01.56, Cat

KeyboardInterrupt: 

Ȇ.




In [ ]:
print(infer(val_data[55]))

In [ ]:
print(infer(val_data[55]))

In [26]:
print(val_data_ans[55])

3.53


In [ ]:
input_string = infer(train_data[4])
start_index = input_string.find("Answer: ") + len("Answer: ")

# Extract the first 5 characters from that point
result = input_string[start_index:start_index + 5]

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


In [ ]:
print(result)

03.52


In [ ]:
test_data[0]

'<|startoftext|>CatȊ, 08.31, Catȋ, 06.62, Catȍ, Cat_Ȑ, 07.46, CatȎ, 07.88, CatȐ, 06.63, CatȖ, 08.78, CatȘ, 06.55, Cat_ț, Catț, 08.18. Find min of categories Catȋ, Catț, CatȘ and Catȍ. Answer: '

In [30]:
test_list = [infer(test_data[i]) for i in range(100)]

In [45]:
print(infer2(val_data[55]))

CatǶ, 02.61, CatǷ, 02.41, Cat+ȉ, Cat-ȁ, CatǺ, 02.33, CatǼ, 02.14, CatȀ, 02.68, Catȁ, 02.30, CatȂ, 02.06, Catȉ, 02.63. Find min of categories CatǺ, Catȁ, CatǷ and CatǼ. Answer: Ȃ
